In [3]:
import os
import fitz
import faiss
import numpy as np
import re
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LOAD DOCS

In [8]:
DOCS_DIR = r"docs"
CHUNK_SIZE = 700
OVERLAP = 300
K_RETRIEVE = 3
SIMILARITY_THRESHOLD = 0.8
TOP_N = 10 

In [9]:
documents = []
for filename in os.listdir(DOCS_DIR):
    if filename.endswith(".pdf"):
        path = os.path.join(DOCS_DIR, filename)
        doc = fitz.open(path)
        text = ""
        for page in doc:
            text += page.get_text()
        documents.append({"name": filename, "text": text})

print(f"Loaded {len(documents)} documents")

Loaded 11 documents


 **Chunking**

In [10]:
def chunk_text(text, chunk_size=350, overlap=150):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

all_chunks = []
chunk_sources = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    all_chunks.extend(chunks)
    chunk_sources.extend([doc["name"]] * len(chunks))
print(f"Total chunks: {len(all_chunks)}")

Total chunks: 210


 **Embedding**

In [11]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(all_chunks, convert_to_numpy=True)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True) 

c:\Users\Vignesh\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\Vignesh\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [12]:
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors.")

FAISS index built with 210 vectors.


In [13]:

cross_encoder_model= CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")


Query Retrieval   Cross-Encoder Reranking  and LLM Answer Generation

In [ ]:

def retrieve(query, k=3, top_n=10):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)
    query_emb = query_emb / np.linalg.norm(query_emb)
    D, I = index.search(query_emb, top_n)
    
    candidates = [(all_chunks[i], chunk_sources[i]) for i in I[0]]
  
    cross_inputs = [(query, c[0]) for c in candidates]
    scores = cross_encoder_model.predict(cross_inputs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    
    top_chunks, top_sources = zip(*[r[0] for r in ranked[:k]])
    return list(top_chunks), list(top_sources)



llm_model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

def classify_question(query: str) -> str:
    """Simple rule-based classifier for question type."""
    q = query.lower()
    if "purpose" in q or "function" in q or "role" in q:
        return "purpose"
    elif "principle" in q or "operation" in q or "how" in q:
        return "operation"
    elif "maintenance" in q or "check" in q or "service" in q:
        return "maintenance"
    elif "pressure" in q or "differential" in q:
        return "requirement"
    elif "remove" in q or "separate" in q:
        return "removal"
    elif "piping" in q or "arrangement" in q:
        return "arrangement"
    else:
        return "general"


def generate_answer(query, k=3, similarity_threshold=0.6):
   
    chunks, sources = retrieve(query, k=k)
    if not chunks:
        return "Sorry, no relevant information found."
    

    context_text = "\n\n".join([f"Document: {s}\n{c}" for c, s in zip(chunks, sources)])
    
 
    prompt = f"""
You are a technical assistant.
Answer the question using ONLY the information provided below.
Rephrase the answer concisely.
Do NOT include information beyond the context.
If no relevant information, reply: "Sorry, no relevant information found."

Question: {query}
Context:
{context_text}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    output_ids = model.generate(**inputs, max_length=300, num_beams=5, early_stopping=True)
    generated_answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    
    scores = [semantic_similarity(generated_answer, c) for c in chunks]
    max_score = max(scores)

   
    if max_score < similarity_threshold:
        idx = scores.index(max_score)
        fallback_answer = f"{chunks[idx]} [{sources[idx]}]"
        return fallback_answer

   
    return f"{generated_answer} [{sources[0]}]"

from sentence_transformers import SentenceTransformer, util
semantic_model = SentenceTransformer('all-mpnet-base-v2')
def semantic_similarity(a, b):
    """Compute cosine similarity between two texts."""
    emb_a = semantic_model.encode(a, convert_to_tensor=True)
    emb_b = semantic_model.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb_a, emb_b).item()


In [ ]:
import pandas as pd

def evaluate_pipeline(questions, k=3, threshold=0.7, csv_path='evaluation.csv'):
    """
    Evaluate both LLM grounding and retrieval support for each question.
    Save combined results to a single CSV file.
    """
    results = []
    correct_max = 0
    correct_avg = 0
    retrieval_hits = 0
    total_questions = len(questions)

    for q in questions:
      
        generated = generate_answer(q, k=k)

       
        chunks, _ = retrieve(q, k=k)

        if not chunks:
            grounding_max, grounding_avg, retrieval_max = 0, 0, 0
        else:
            scores = [semantic_similarity(generated, c) for c in chunks]
            grounding_max = max(scores)
            grounding_avg = sum(scores) / len(scores)
            retrieval_max = max(scores)

     
        if grounding_max >= threshold:
            correct_max += 1
        if grounding_avg >= threshold:
            correct_avg += 1
        if retrieval_max >= threshold:
            retrieval_hits += 1

        results.append({
            "question": q,
            "generated_answer": generated,
            "grounding_max": grounding_max,
            "grounding_avg": grounding_avg,
            "retrieval_max": retrieval_max
        })

        print(f"Q: {q}")
        print(f"Generated Answer: {generated}")
        print(f"Grounding Score (max): {grounding_max:.2f}")
        print(f"Grounding Score (avg): {grounding_avg:.2f}")
        print(f"Max Retrieval Support: {retrieval_max:.2f}\n")


    precision_max = correct_max / total_questions
    precision_avg = correct_avg / total_questions
    recall = retrieval_hits / total_questions

    print("===== Overall Metrics =====")
    print(f"Grounding Precision@{threshold} (max): {precision_max:.2f}")
    print(f"Grounding Precision@{threshold} (avg): {precision_avg:.2f}")
    print(f"Retrieval Recall@{k} (threshold={threshold}): {recall:.2f}")

   
    df = pd.DataFrame(results)
    df.to_csv(csv_path, index=False)
    print(f"Combined evaluation results saved to {csv_path}")

    return df, precision_max, precision_avg, recall

In [16]:


query = "What is the purpose of the cyclone separator?"

answer = generate_answer(query)
print("Q:", query)
print("A:", answer)



Q: What is the purpose of the cyclone separator?
A: to create an eddy motion during which dust particles are separated from the air or gas. [654b45bb8292f586313470.pdf]


Use any one of the test data.

In [17]:
test_questions2 = [
    "What is the principle of operation for Armstrong’s cyclone separator?",
    "What maintenance is required for Armstrong’s cyclone separator?",
    "What pressure differential is required for suitable cyclone separator operation?",
    "What does the cyclone separator remove from the flushing liquid?",
    "What is the typical piping arrangement for the Armstrong cyclone separator?",
    "What should be periodically checked to ensure cyclone separator is functioning properly?"
]


In [18]:
test_questions = [
    "What is the first step before using a cyclonic separator?",
    "How is the collecting efficiency related to water gauge?",
    "What role does particle density play in cyclone operation?",
    "How should cyclones be supported during installation?",
    "What daily maintenance is recommended for cyclones?",
    "What precautions should be taken to avoid electric shock?",
    "How is the cyclone separator designed to operate safely?",
    "What is the principle of operation of the Armstrong cyclone?",
    "How does the TLV cyclone separator remove contaminants?",
    "What are the requirements for electrical installation of the cyclone?",
    "How do you adjust cyclone airflow in the Kice system?",
    "What is a cyclone's cut point in classification?",
    "How does a cyclone operate to separate particulates?",
    "What are the differences between cyclone types G, M, H, and MC?",
    "What are the recommended inspection intervals for cyclone parts?",
    "How should operators respond when cyclone efficiency drops?",
    "How to handle malfunctions in cyclone operation?",
    "What is the role of the cyclone's vortex finder?",
    "How to properly install the cyclone on site?",
    "What safety devices must be used during cyclone installation?",
    "How to maintain the cyclone motor and belt drive?",
    "What should be done before starting cyclone operation?",
    "What are the emergency procedures for cyclone explosion venting?",
    "How to mitigate cyclone wear in abrasive environments?",
    "What documents accompany the cyclone for installation and maintenance?",
    "What are the personal protective equipment requirements?",
    "How often should cyclone dust containers be emptied?",
    "What are the main hazards associated with cyclone use?",
    "How to safely dismantle and dispose of cyclone components?",
    "What steps should be taken after receiving a cyclone storm warning?",
    "How is cyclone-related weather information communicated to ships?",
    "What are the responsibilities of port authorities during cyclone events?",
    "How is cyclone separator performance affected by particle shape?",
    "What is the purpose of the cyclone’s cone section?",
    "What is the impact of feed pressure on cyclone operation?",
    "What causes cyclone spigot blockage and how to prevent it?",
    "What are common problems caused by fluctuating feed in cyclones?",
    "How to optimize cyclone efficiency for different materials?",
    "What training must personnel have for cyclone installation and repair?",
    "How can cyclone dust be safely removed and handled?",
    "What are the installation recommendations for cyclone ducts and transitions?",
    "How to interpret cyclone performance data and adjust operation accordingly?"
]


In [19]:
df, prec_max, prec_avg, rec = evaluate_pipeline(test_questions, k=3, threshold=0.7)



Q: What is the first step before using a cyclonic separator?
Generated Answer: accordance with the manual and observe all instructions and directions given.  Never change the order of the steps to perform.  Always keep a copy of this manual with the product.  This product should be used only by authorized, trained personnel.  This manual is intended to cover a range of units. Therefore some aspects may not apply to your particular unit. 4 CYCLONIC SEPARATORS (CYCLONES) - GENERAL DESCRIPTION Much has been written on the subject of passing dust through a unit having cyclone proportions. In analysing the numerous references which are available, the reader finds great difficulty in isolating basic facts on which he can begin to build his understanding. The first rule which is generally accepted is the simple empirically proved fact that the end product, which can be termed as ‘collecting efficiency’, is related to energy expended. In more basic terms we set out the rough yardstick ‘col

In [20]:

print("===== Final Evaluation Metrics =====")
print(f"Grounding Precision@0.7 (max): {prec_max:.2f}")
print(f"Grounding Precision@0.7 (avg): {prec_avg:.2f}")
print(f"Retrieval Recall@3: {rec:.2f}")


===== Final Evaluation Metrics =====
Grounding Precision@0.7 (max): 0.93
Grounding Precision@0.7 (avg): 0.71
Retrieval Recall@3: 0.93
